# Audio Restoration and Enhancement
This notebook covers the four projects in this folder's README:
1. **Noise reduction and speech enhancement**: spectral subtraction and Wiener filtering in the STFT domain (section 1).
2. **Adaptive noise cancellation**: an LMS adaptive filter that learns to remove noise picked up by a second microphone (section 2).
3. **De-clicking**: detecting clicks with a prediction model and filling them in (section 3).
4. **De-clipping**: rebuilding the tops of waveforms that were clipped (section 4).

Every method is tested on a clean signal that is damaged on purpose, so the result can be measured against the original. The speech is synthetic: vowels made with the source-filter model of the voice (`audiodsp.synthetic_vowels`).

The noise reduction and adaptive filter code lives in `audiodsp/restoration.py`, because the telephone and speech capstones in `09_Capstone_Projects` reuse it. Its source is shown below where it is used.

In [ ]:
import inspect
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate, linalg, signal
from IPython import display
import ipywidgets as widgets
from ipywidgets import interactive
import audiodsp
from audiodsp import nlms, noise_reduction, pluck, synthetic_vowels

%matplotlib inline
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (14, 5)

rng = np.random.default_rng(0)

def play(x, label, fs):
    print(f'{label}:')
    display.display(display.Audio(np.clip(x, -1, 1), rate=fs, normalize=False))

def snr_db(clean, test):
    return 10 * np.log10(np.mean(clean**2) / np.mean((test - clean)**2))

def show_source(func):
    display.display(display.Code(inspect.getsource(func), language='python'))

FS_SPEECH = 16000
speech, speech_active = synthetic_vowels('aeiouaei', sample_rate=FS_SPEECH)

FS = 44100
melody = np.zeros(4 * FS)
for i, f in enumerate([220, 277.2, 329.6, 440, 554.4, 440, 329.6, 277.2, 220, 329.6, 440, 659.3, 880, 659.3, 440, 220]):
    start = int(i * 0.25 * FS)
    melody[start:start + int(0.25 * FS)] += pluck(f, 0.25, FS)
melody *= 0.9 / np.max(np.abs(melody))
# Every real recording has some background noise. Without it, a synthetic signal is so predictable
# that the click detector in section 3 would flag the tiniest irregularity.
melody += rng.normal(0, 10 ** (-50 / 20), len(melody))

## 1. Noise reduction
Broadband noise (hiss, fan noise, traffic) overlaps the signal in both time and frequency, so no fixed filter can remove it. STFT-based noise reduction instead applies a different **gain** to every time-frequency bin: close to 1 where the signal dominates, small where noise dominates.

It needs an estimate of the noise spectrum. Here it comes from the first 0.4 s of the recording, before the speech starts, as an editor would select a noise-only stretch. With noise power $N(k)$ in bin $k$, and the noisy signal's power $|X(m,k)|^2$ in frame $m$:
- **Spectral subtraction** removes the noise power: $G = \sqrt{\max\!\left(1 - N / |X|^2,\ G_{min}^2\right)}$.
- The **Wiener filter** uses $G = \xi / (1 + \xi)$, where $\xi$ is the signal-to-noise ratio of the bin. Estimating $\xi$ from one noisy frame is unreliable, and that unreliability causes **musical noise**: isolated noise bins that randomly survive, heard as short twittering tones. The **decision-directed** estimate (Ephraim and Malah, 1984) mixes in the previous frame's cleaned result, which smooths $\xi$ over time and largely removes the effect.
- The **spectral gate** from `02_Frequency_Domain_Analysis` is the crudest version: gain 1 or 0.

All three keep a minimum gain $G_{min}$ (the **floor**, 0.1 here, which is -20 dB) so a little noise remains. Removing it completely makes the gaps sound unnaturally dead and makes artifacts worse.

In [ ]:
show_source(noise_reduction)

In [ ]:
def spectral_gate(x, noise, fs, nperseg=512):
    stft_args = dict(fs=fs, nperseg=nperseg, noverlap=3 * nperseg // 4)
    _, _, X = signal.stft(x, **stft_args)
    _, _, N = signal.stft(noise, **stft_args)
    threshold = 3 * np.sqrt(np.mean(np.abs(N)**2, axis=1, keepdims=True))
    _, y = signal.istft(X * (np.abs(X) > threshold), **stft_args)
    return y[:len(x)]

def noise_reduction_demo(snr_in=5):
    noise = rng.normal(0, 1, len(speech))
    noise *= np.sqrt(np.mean(speech[speech_active]**2) / np.mean(noise**2)) * 10 ** (-snr_in / 20)
    noisy = speech + noise
    noise_only = noisy[:int(0.4 * FS_SPEECH)]
    results = {
        'noisy': noisy,
        'spectral gate': spectral_gate(noisy, noise_only, FS_SPEECH),
        'spectral subtraction': noise_reduction(noisy, noise_only, FS_SPEECH, method='subtraction'),
        'Wiener, decision-directed': noise_reduction(noisy, noise_only, FS_SPEECH, method='wiener'),
    }
    fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
    for ax, (name, y) in zip(axes, results.items()):
        f, t, Sxx = signal.spectrogram(y, FS_SPEECH, nperseg=512, noverlap=384)
        ax.pcolormesh(t, f, 10 * np.log10(Sxx + 1e-12), shading='auto', vmin=-110, vmax=-30)
        ax.set_title(f'{name}\nSNR {snr_db(speech, y):.1f} dB')
        ax.set_xlabel('Time (s)')
    axes[0].set_ylabel('Frequency (Hz)')
    plt.tight_layout()
    plt.show()
    for name, y in results.items():
        play(y, name, FS_SPEECH)

display.display(interactive(
    noise_reduction_demo,
    snr_in=widgets.IntSlider(value=5, min=-5, max=20, step=1, description='Input SNR (dB):'),
))

Listen for the difference between subtraction and Wiener filtering in the gaps between vowels: subtraction leaves twittering musical noise, the decision-directed Wiener filter a quieter, steady hiss.

SNR is only a rough guide here. A method that removes more noise can still sound worse, if its artifacts are more noticeable than the noise it removed.

## 2. Adaptive noise cancellation
When a second microphone can pick up the noise source on its own, the noise can be removed far more completely. The main microphone records speech plus noise that has travelled along some unknown acoustic path. The reference microphone, placed near the noise source, records the noise alone. An **adaptive filter** learns the unknown path, predicts the noise in the main microphone from the reference, and subtracts it. What is left is the speech, because nothing in the reference is correlated with it.

The **normalized LMS** (NLMS) algorithm adjusts the filter weights $w$ after every sample, by a small step in the direction that reduces the error $e[n]$:

$$e[n] = d[n] - w^T x[n], \qquad w \leftarrow w + \mu\, \frac{e[n]\, x[n]}{\lVert x[n] \rVert^2}$$

where $d[n]$ is the main microphone and $x[n]$ holds the most recent reference samples. The step size $\mu$ (between 0 and 2) sets a trade-off. A large step adapts quickly, but the speech in $e[n]$ keeps pushing the weights around, which leaves residual noise. The excess error power is about $\mu / (2 - \mu)$ times the speech power. A small step is more accurate but slower to adapt.

The same algorithm cancels the echo in phone calls (`09_Capstone_Projects`).

In [ ]:
show_source(nlms)

In [ ]:
path = rng.normal(0, 1, 32) * np.exp(-np.arange(32) / 6)    # unknown acoustic path from noise source to main microphone
path /= np.linalg.norm(path)
noise_source = rng.normal(0, 0.3, len(speech))
main_mic = speech + np.convolve(noise_source, path)[:len(speech)]
reference_mic = noise_source

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
results = {}
for step in [1.0, 0.3, 0.05, 0.01]:
    cleaned, weights = nlms(reference_mic, main_mic, n_taps=48, step=step)
    results[step] = cleaned
    residual_db = 10 * np.log10(signal.convolve((cleaned - speech)**2, np.ones(1600) / 1600, mode='same') + 1e-12)
    axes[0].plot(np.arange(len(speech)) / FS_SPEECH, residual_db, label=f'mu = {step}: SNR {snr_db(speech[len(speech) // 2:], cleaned[len(speech) // 2:]):.1f} dB')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Residual noise power (dB)')
axes[0].set_title('Convergence (SNR measured over the second half)')
axes[0].legend(fontsize=9)
_, weights = nlms(reference_mic, main_mic, n_taps=48, step=0.05)
axes[1].stem(path, linefmt='C0-', markerfmt='C0o', basefmt=' ', label='true path')
axes[1].plot(weights, 'rx', label='learned weights (mu = 0.05)')
axes[1].set_xlabel('Tap')
axes[1].set_title('The filter learns the acoustic path')
axes[1].legend()
plt.tight_layout()
plt.show()
print(f'Main microphone SNR: {snr_db(speech, main_mic):.1f} dB')
play(main_mic, 'Main microphone', FS_SPEECH)
play(results[0.05], 'After adaptive noise cancellation (mu = 0.05)', FS_SPEECH)

## 3. De-clicking
Scratches on vinyl records and digital transmission errors produce **clicks**: short bursts, a few samples to a millisecond long, much larger than the audio around them.

**Detection.** Music is predictable from its recent past, a click is not. A **linear prediction** model (autoregressive, AR) predicts each sample from the previous $p$ samples:

$$\hat x[n] = -\sum_{i=1}^{p} a_i\, x[n-i]$$

The prediction error is small on music and jumps at every click. The coefficients $a_i$ come from the signal's autocorrelation (the same method speech codecs use, solved here with `scipy.linalg.solve_toeplitz`). Samples whose prediction error exceeds 8 robust standard deviations are marked as damaged, together with a few neighbors.

**Repair.** Each damaged stretch is replaced with the values that best fit the same AR model, given the good samples on both sides: the **least-squares AR interpolator** (Vaseghi and Rayner, 1990). The missing samples are chosen to make the prediction errors around the gap as small as possible, which is a linear least-squares problem. For comparison: a **median filter** over the whole signal, which removes clicks but also smooths the music.

In [ ]:
def lpc(x, order):
    """AR coefficients a[0..order] (a[0] = 1) from the autocorrelation method."""
    r = np.correlate(x, x, mode='full')[len(x) - 1:len(x) + order] / len(x)
    r[0] *= 1 + 1e-9                                       # tiny regularization
    return np.concatenate(([1.0], linalg.solve_toeplitz(r[:order], -r[1:order + 1])))

def detect_clicks(x, order=20, threshold=8.0, widen=3, block=4096):
    """Mark samples with large AR prediction error, block by block, as damaged."""
    damaged = np.zeros(len(x), dtype=bool)
    for start in range(0, len(x), block):
        segment = x[max(0, start - order):start + block]
        a = lpc(segment, order)
        error = signal.lfilter(a, [1.0], segment)[min(start, order):]
        sigma = 1.4826 * np.median(np.abs(error))         # robust standard deviation (from the median)
        damaged[start:start + len(error)] |= np.abs(error) > threshold * sigma
    return np.convolve(damaged, np.ones(2 * widen + 1), mode='same') > 0

def lsar_interpolate(x, damaged, order=20):
    """Least-squares AR interpolation of every run of damaged samples."""
    y = x.copy()
    runs = np.flatnonzero(np.diff(np.concatenate(([0], damaged.astype(int), [0]))))
    for start, stop in zip(runs[::2], runs[1::2]):
        lo, hi = max(0, start - 3 * order), min(len(x), stop + 3 * order)
        segment = y[lo:hi]
        a = lpc(np.concatenate((segment[:start - lo], segment[stop - lo:])), order)
        # Prediction error e = A @ segment, for every n with a full set of past samples
        A = linalg.toeplitz(np.r_[a[-1], np.zeros(len(segment) - order - 1)],
                            np.r_[a[::-1], np.zeros(len(segment) - order - 1)])
        unknown = np.zeros(len(segment), dtype=bool)
        unknown[start - lo:stop - lo] = True
        solution, *_ = np.linalg.lstsq(A[:, unknown], -A[:, ~unknown] @ segment[~unknown], rcond=None)
        y[start:stop] = solution
    return y

clicky = melody.copy()
true_clicks = np.zeros(len(melody), dtype=bool)
for position in rng.choice(len(melody) - 50, 40, replace=False):
    length = rng.integers(1, 20)
    clicky[position:position + length] += rng.choice([-1, 1]) * rng.uniform(0.2, 0.6) * np.exp(-np.arange(length) / 4)
    true_clicks[position:position + length] = True

damaged = detect_clicks(clicky)
repaired = lsar_interpolate(clicky, damaged)
median_filtered = signal.medfilt(clicky, 9)

found = sum(damaged[p] for p in np.flatnonzero(np.diff(np.concatenate(([0], true_clicks.astype(int)))) == 1))
print(f'Clicks found: {found} of 40. Samples marked: {damaged.sum()} ({100 * damaged.mean():.2f}% of the signal)')
for name, y in [('with clicks', clicky), ('median filter (9 samples)', median_filtered), ('detect + LSAR interpolation', repaired)]:
    print(f'{name:<30} SNR {snr_db(melody, y):6.1f} dB')

worst = np.flatnonzero(true_clicks)[0]
view = slice(worst - 300, worst + 300)
fig, ax = plt.subplots(figsize=(14, 4))
t_ms = np.arange(len(melody))[view] / FS * 1000
ax.plot(t_ms, clicky[view], color='0.7', label='with click')
ax.plot(t_ms, melody[view], 'k', linewidth=2, label='original')
ax.plot(t_ms, repaired[view], 'r--', label='repaired')
ax.set_xlabel('Time (ms)')
ax.legend(loc='upper right')
plt.show()
play(clicky, 'With clicks', FS)
play(median_filtered, 'Median filter', FS)
play(repaired, 'Detected and interpolated', FS)

## 4. De-clipping
A signal recorded too loud is **clipped**: every sample beyond the converter's range is stored as the maximum value, flattening the tops of the waveform and adding harsh distortion. The information above the clipping level is gone, but clipped samples are easy to find (they sit exactly at the limit), and the waveform around them shows where it was heading.

The repair below fits a smooth curve through the good samples on both sides of each clipped run, using a **cubic spline** over a window of good samples, then makes sure the result is at least as large as the clipping level (the true signal was larger, or it would not have clipped).

In [ ]:
def declip(x, clip_level, context=40):
    clipped = np.abs(x) >= clip_level * 0.9999
    y = x.copy()
    runs = np.flatnonzero(np.diff(np.concatenate(([0], clipped.astype(int), [0]))))
    for start, stop in zip(runs[::2], runs[1::2]):
        lo, hi = max(0, start - context), min(len(x), stop + context)
        idx = np.arange(lo, hi)
        good = ~clipped[lo:hi]
        if good.sum() < 4:
            continue
        spline = interpolate.CubicSpline(idx[good], x[lo:hi][good])
        estimate = spline(np.arange(start, stop))
        sign = np.sign(x[start])
        y[start:stop] = sign * np.maximum(sign * estimate, clip_level)
    return y, clipped

def declip_demo(clip_db=-6):
    clip_level = 0.9 * 10 ** (clip_db / 20)
    damaged_signal = np.clip(melody, -clip_level, clip_level)
    restored, clipped = declip(damaged_signal, clip_level)
    region = clipped
    print(f'Clipped samples: {100 * clipped.mean():.2f}%')
    for name, y in [('clipped', damaged_signal), ('de-clipped', restored)]:
        print(f'{name:<11} SNR {snr_db(melody, y):6.1f} dB overall, '
              f'error in clipped samples {10 * np.log10(np.mean((y[region] - melody[region])**2) / np.mean(melody[region]**2)):6.1f} dB')
    peak = np.argmax(np.abs(melody))
    view = slice(peak - 200, peak + 200)
    t_ms = np.arange(len(melody))[view] / FS * 1000
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t_ms, melody[view], 'k', linewidth=2, label='original')
    ax.plot(t_ms, damaged_signal[view], color='0.6', label='clipped')
    ax.plot(t_ms, restored[view], 'r--', label='de-clipped')
    ax.set_xlabel('Time (ms)')
    ax.legend(loc='upper right')
    plt.show()
    play(damaged_signal, 'Clipped', FS)
    play(restored * 0.9 / np.max(np.abs(restored)), 'De-clipped (rescaled to avoid clipping again)', FS)

display.display(interactive(
    declip_demo,
    clip_db=widgets.IntSlider(value=-3, min=-12, max=-1, step=1, description='Clip at (dB):'),
))

The spline works well for mild clipping, where each clipped run is short: at -3 dB it reduces the error in the clipped samples by about 15 dB. At -6 dB it barely helps, and with heavier clipping it makes the error worse. When long stretches are clipped, the spline has too little to go on, and heavier methods are needed: AR or sparse-signal models that use many periods of the waveform, as in commercial restoration tools.